In [7]:
#Class dataset for MRI 
import os
import pickle
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import cv2  # used for resizing

class MRISliceDataset(Dataset):
    def __init__(self, root_dir, metadata_csv, transform=None, max_files=50, target_size=(320, 320)):
        self.root_dir = root_dir
        self.metadata = pd.read_csv(metadata_csv)
        self.transform = transform
        self.target_size = target_size

        # Keep only first N volumes
        unique_volumes = self.metadata["volumeFilename"].unique()[:max_files]
        self.metadata = self.metadata[self.metadata["volumeFilename"].isin(unique_volumes)]

        self.samples = []
        for _, row in self.metadata.iterrows():
            vol_path = os.path.join(root_dir, row["volumeFilename"])
            if not os.path.exists(vol_path):
                continue
            try:
                with open(vol_path, "rb") as f:
                    volume = pickle.load(f)
                num_slices = volume.shape[0]
            except Exception as e:
                print(f"⚠️ Skipping {vol_path}: {e}")
                continue

            roi_start = int(row["roiZ"])
            roi_end = roi_start + int(row["roiDepth"]) - 1

            for i in range(num_slices):
                label = 1 if roi_start <= i <= roi_end else 0
                self.samples.append((vol_path, i, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vol_path, slice_idx, label = self.samples[idx]

        with open(vol_path, "rb") as f:
            volume = pickle.load(f)

        image = volume[slice_idx].astype(np.float32)
        image = image / np.max(image) if np.max(image) > 0 else image

        # ✅ Resize to target_size using OpenCV (H, W)
        image = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)

        # Add channel dimension
        image = np.expand_dims(image, axis=0)
        image = torch.tensor(image, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


In [6]:
#LeNet Model 
import torch
import torch.nn as nn
import torch.nn.functional as F
#     para = {
#         'input_image_size': (320, 320),
#         'input_channel': 1,
#         'number_of_conv_layer':2,
#         'number_of_fc_layer':2,
#         'num_classes':2,
#     }
class LeNet(nn.Module):
    def __init__(self, para):
        super(LeNet,self).__init__()
        self.para=para
        self.input_image_size=para['input_image_size']
        self.input_channel=para['input_channel']
        self.num_conv_layer=para['number_of_conv_layer']
        self.num_fc_layer=para['number_of_fc_layer']
        self.fc_feateure=para['fc_feature']
        self.conv_channels=para['conv_channels']
        self.Conv_layer=nn.ModuleList()
        self.FC_layer=nn.ModuleList()

        self.build_conv_layer()
        self.build_fc_layer()
    def build_conv_layer(self):
        for i in range(self.num_conv_layer):
            in_channels=self.input_channel if i==0 else self.conv_channels[i-1]
            out_channels=self.conv_channels[i]
            conv=nn.Sequential(nn.Conv2d(in_channels,out_channels,kernel_size=7,stride=1,padding=2),
                               nn.BatchNorm2d(out_channels),
                               nn.ReLU(),
                                nn.MaxPool2d(kernel_size=2,stride=2))
            self.Conv_layer.append(conv)
    def forward_convs(self, x):
        for conv in self.Conv_layer:
            x = conv(x)
        return x
    def build_fc_layer(self):
        in_features=0
        with torch.no_grad():
            x=torch.randn(1,self.input_channel,self.input_image_size[0],self.input_image_size[1])
            x=self.forward_convs(x)
            in_features=x.flatten(1).shape[1]
        fc=nn.Sequential(nn.Linear(in_features,self.fc_feateure[0]),
                         nn.ReLU(),
                         nn.Linear(self.fc_feateure[0],self.fc_feateure[1]),
                         nn.Softmax(dim=1))
        self.FC_layer.append(fc)
    def forward_fc(self, x):
        x = x.flatten(1)
        for fc in self.FC_layer:
            x = fc(x)
        return x

    def forward(self, x):
        x = self.forward_convs(x)
        x = self.forward_fc(x)
        return x
    def summary(self):
        print("="*40)
        print(f"{'Layer':<15}{'Output Shape':<20}{'Details'}")
        print("="*40)
        for i, conv in enumerate(self.Conv_layer):
            print(f"Conv{i+1:<10} {str(conv):<20}")
        for i, fc in enumerate(self.FC_layer):
            print(f"FC{i+1:<10} {str(fc):<20}")
        print("="*40)

if __name__ == "__main__":
    para = {
        'input_image_size': (320, 320),
        'input_channel': 1,
        'number_of_conv_layer':2,
        'number_of_fc_layer':2,
        'num_classes':2,
        'conv_channels': [6, 16],
        'fc_feature':[84,2]
    }
    model = LeNet(para)
    x = torch.randn(1, 1, 320, 320)
    y = model(x)
    print("Output shape:", y.shape)
    print(f"Predicted class:{y}")
    model.summary()

Output shape: torch.Size([1, 2])
Predicted class:tensor([[0.6279, 0.3721]], grad_fn=<SoftmaxBackward0>)
Layer          Output Shape        Details
Conv1          Sequential(
  (0): Conv2d(1, 6, kernel_size=(7, 7), stride=(1, 1), padding=(2, 2))
  (1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
Conv2          Sequential(
  (0): Conv2d(6, 16, kernel_size=(7, 7), stride=(1, 1), padding=(2, 2))
  (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
FC1          Sequential(
  (0): Linear(in_features=97344, out_features=84, bias=True)
  (1): ReLU()
  (2): Linear(in_features=84, out_features=2, bias=True)
  (3): Softmax(dim=1)
)


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

class Trainer:
    def __init__(self, model, train_dataset, val_dataset=None,
                 batch_size=64, lr=1e-4, num_workers=0, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        # Dataloaders
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                       shuffle=True, num_workers=num_workers)
        self.val_loader = None
        if val_dataset is not None:
            self.val_loader = DataLoader(val_dataset, batch_size=batch_size,
                                         shuffle=True, num_workers=num_workers)
        # Loss and optimizer
        self.criterion = nn.CrossEntropyLoss()  # binary classification
        self.optimizer = torch.optim.SGD(self.model.parameters(),lr=lr)

    def train_epoch(self):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        loop = tqdm(self.train_loader, desc="Training", leave=False)
        for images, labels in loop:
            images, labels = images.to(self.device), labels.to(self.device)
            
            # Forward
            outputs = self.model(images).squeeze(1)  # [B] or [B,1]
            loss = self.criterion(outputs, labels)

            # Backward
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            # Metrics
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels.long()).sum().item()
            total += labels.size(0)
            loop.set_postfix(loss=loss.item())

        avg_loss = running_loss / total
        acc = correct / total
        return avg_loss, acc

    def validate_epoch(self):
        if self.val_loader is None:
            return None, None

        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            loop = tqdm(self.val_loader, desc="Validating", leave=False)
            for images, labels in loop:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images).squeeze(1)
                loss = self.criterion(outputs, labels)
                print(outputs.shape,outputs.dtype)
                print(labels.shape,labels.dtype)
                running_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels.long()).sum().item()
                total += labels.size(0)

        avg_loss = running_loss / total
        acc = correct / total
        return avg_loss, acc

    def fit(self, epochs=2):
        print(f"Training on device: {self.device}")
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc = self.validate_epoch() if self.val_loader else (None, None)

            msg = f"Epoch [{epoch}/{epochs}] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}"
            if val_loss is not None:
                msg += f" | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
            print(msg)


In [12]:
from torch.utils.data import random_split

full_dataset = MRISliceDataset("./MRI_dataset/volumetric_data/", "./MRI_dataset/metadata.csv")
train_size = int(0.7 * len(full_dataset))
print(train_size)
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
para = {
        'input_channel': 1,
        'number_of_conv_layer': 2,
        'number_of_FC_layer': 2,
        'FC_features': [84, 2],
        'output_channel': [6, 16],
        'Kernel_size': [7, 5],
        'Stride': [1, 1],
        'Padding': [0, 0],
        'Activation_Func': ['ReLU', 'ReLU'],
        'Pooling_type': ['MaxPool', 'MaxPool'],
        'FC_Activation_Func': ['ReLU', 'None'], 
        'use_batchnorm': True,
    }
model = LeNet(para)
print(sum(p.numel() for p in model.parameters()))  # should be > 0
trainer = Trainer(model, train_ds, val_ds, batch_size=8, lr=1e-4)
trainer.fit(epochs=10)


1019
7765958
Training on device: cuda


Validating:   4%|▎         | 2/55 [00:00<00:07,  6.89it/s]              

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


Validating:   7%|▋         | 4/55 [00:00<00:07,  7.06it/s]

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


...

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


Validating:  98%|█████████▊| 54/55 [00:08<00:00,  6.84it/s]

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


torch.Size([6, 2]) torch.float32
torch.Size([6]) torch.int64
Epoch [1/10] | Train Loss: 0.3123, Acc: 0.8901 | Val Loss: 0.2348, Val Acc: 0.9132


KeyboardInterrupt: 